In [1]:
import datetime
import os

import disturbancemonitor as dm
from sentinelhub import SHConfig

/home/jviehweger/Documents/Projects/2024/change-detection/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
config = SHConfig()

os.environ["SH_CLIENT_ID"] = config.sh_client_id
os.environ["SH_CLIENT_SECRET"] = config.sh_client_secret

input_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [-96.643464, 40.892987],
            [-96.633766, 40.892987],
            [-96.633766, 40.900708],
            [-96.643464, 40.900708],
            [-96.643464, 40.892987],
        ]
    ],
}

monitor = dm.start_monitor(
    name="another-monitor",
    monitoring_start=datetime.date(2021, 2, 1),
    geometry=input_geojson,
    resolution=0.005,
    bucket_name="some-monitor-yqtoxmy7",
)

0/6 Initializing model
1/6 Creating bucket
2/6 Fitting model
3/6 Writing model to bucket
4/6 Ingesting model to SH
Waiting for collection to finish ingestion
Ingested
5/6 Computing metric
6/6 Writing metric to bucket


In [4]:
config = SHConfig()

os.environ["SH_CLIENT_ID"] = config.sh_client_id
os.environ["SH_CLIENT_SECRET"] = config.sh_client_secret
monitor = dm.load_monitor("another-monitor")

In [5]:
process_data = monitor.monitor(end=datetime.date(2021, 4, 1))

In [6]:
binary = process_data.content

In [7]:
import datetime
import os

import xarray as xr
from rasterio.io import MemoryFile

with MemoryFile(binary).open() as dataset:
    metrics_ds = xr.open_dataset(dataset, band_as_variable=True, engine="rasterio")
order = ["y", "x"]
reordered_indexes = {index_name: metrics_ds.indexes[index_name] for index_name in order}

In [14]:
reordered_indexes

{'y': Index([40.89877775, 40.89491725], dtype='float64', name='y'),
 'x': Index([-96.64103949999999, -96.6361905], dtype='float64', name='x')}

In [13]:
rename = ["disturbedDate", "process"]
{col: new_name for col, new_name in zip(metrics_ds, rename)}

{'band_1': 'disturbedDate', 'band_2': 'process'}

In [11]:
from dataclasses import dataclass, field


@dataclass
class MonitorParameters:
    name: str
    monitoring_start: str
    geometry: dict
    resolution: tuple
    datasource: str
    harmonics: int = 2
    inputs: list[str] = field(default_factory=lambda: ["NDVI"])
    metric: str = "RMSE"
    last_monitored: datetime.date = None
    sensitivity: int = 5
    boundary: int = 5

    def __post_init__(self):
        if self.last_monitored is None:
            self.last_monitored = self.monitoring_start

In [12]:
kwargs = dict(name="hey", monitoring_start=1, geometry={"b": 1}, resolution=(10, 10), datasource="ay")

test = MonitorParameters(**kwargs)

In [8]:
config = SHConfig()

os.environ["SH_CLIENT_ID"] = config.sh_client_id
os.environ["SH_CLIENT_SECRET"] = config.sh_client_secret

input_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [-96.643464, 40.892987],
            [-96.633766, 40.892987],
            [-96.633766, 40.900708],
            [-96.643464, 40.900708],
            [-96.643464, 40.892987],
        ]
    ],
}

dbm = DisturbanceMonitor(
    name="test_monitor",
    bucket_name="sh-change-detection-test",
    monitoring_start="2022-01-01",
    geometry=input_geojson,
    resolution=0.001,
    datasource="ARPS",
)

Setting up bucket
0/5 Initializing model
1/5 Fitting model
2/5 Writing model to bucket
3/5 Ingesting model to SH
Waiting for collection to finish ingestion
Ingested
4/5 Computing metric
5/5 Writing metric to bucket
